# Reparametrization Visual Tests

This notebook tests the functions in `diffusion_strings/reparametrization.py` on several representative and edge cases, and visualizes the results.

In [ ]:
import torch
import matplotlib.pyplot as plt

from diffusion_strings.reparametrization import (
    _segment_lengths,
    _merge_close_points,
    uniform_string_repametrize_rn_linear,
    uniform_string_repametrize_rn_cubic,
    uniform_string_repametrize_so3_linear,
    uniform_string_repametrize_se3_linear,
)
from diffusion_strings.so3 import rotvec_to_rotmat, rotmat_to_theta

torch.set_default_dtype(torch.float64)
plt.rcParams['figure.figsize'] = (7, 5)
plt.rcParams['axes.grid'] = True

## Utilities

In [ ]:
def segment_lengths_rn(points):
    return _segment_lengths(torch.diff(points, dim=0))

def segment_angles_so3(rotations):
    rel = torch.matmul(rotations[:-1].transpose(-1, -2), rotations[1:])
    return rotmat_to_theta(rel)

def print_stats(name, values):
    if values.numel() == 0:
        print(f"{name}: empty")
        return
    print(
        f"{name}: n={values.numel()}, min={values.min().item():.6f}, "
        f"max={values.max().item():.6f}, mean={values.mean().item():.6f}, std={values.std(unbiased=False).item():.6f}"
    )

## 1) Rn linear reparametrization
Test on a non-uniform 2D polyline and check that output segment lengths become near-uniform.

In [ ]:
string = torch.tensor([
    [0.0, 0.0],
    [0.2, 0.0],
    [1.6, 0.8],
    [1.9, 0.9],
    [3.0, 0.9],
])
resampled = uniform_string_repametrize_rn_linear(string, n_new=40)

orig_seg = segment_lengths_rn(string)
new_seg = segment_lengths_rn(resampled)
print_stats('Original segment lengths', orig_seg)
print_stats('Resampled segment lengths', new_seg)

plt.figure()
plt.plot(string[:, 0], string[:, 1], 'o--', label='original')
plt.plot(resampled[:, 0], resampled[:, 1], '.-', label='resampled (linear)')
plt.axis('equal')
plt.legend()
plt.title('Rn linear: geometry')
plt.show()

plt.figure()
plt.plot(orig_seg.numpy(), 'o-', label='original segments')
plt.plot(new_seg.numpy(), '.-', label='resampled segments')
plt.legend()
plt.title('Rn linear: segment length distribution')
plt.xlabel('segment index')
plt.ylabel('length')
plt.show()

## 2) Merge behavior (`_merge_close_points`)
Build a string with very short local segments and visualize before/after merging.

In [ ]:
string_merge = torch.tensor([[0.0], [0.05], [0.10], [0.90], [0.95], [1.0]])
seg = segment_lengths_rn(string_merge)
merged = _merge_close_points(string_merge, seg, merge_tol=0.1)

print('Original points:', string_merge.squeeze().tolist())
print('Merged points:  ', merged.squeeze().tolist())

plt.figure(figsize=(8, 2.6))
plt.plot(string_merge.squeeze().numpy(), torch.zeros(string_merge.shape[0]).numpy(), 'o', label='original')
plt.plot(merged.squeeze().numpy(), torch.ones(merged.shape[0]).numpy(), 'o', label='merged')
plt.yticks([0, 1], ['original', 'merged'])
plt.title('Merge close points')
plt.xlabel('x')
plt.legend()
plt.show()

## 3) Rn cubic vs linear
Use a coarse curved path and compare smoothness and segment uniformity.

In [ ]:
curve = torch.tensor([
    [0.0, 0.0],
    [0.5, 1.0],
    [1.3, 0.2],
    [2.0, 1.2],
    [3.0, 0.8],
])
lin = uniform_string_repametrize_rn_linear(curve, n_new=80)
cub = uniform_string_repametrize_rn_cubic(curve, n_new=80)

seg_lin = segment_lengths_rn(lin)
seg_cub = segment_lengths_rn(cub)
print_stats('Linear segments', seg_lin)
print_stats('Cubic segments', seg_cub)

plt.figure()
plt.plot(curve[:, 0], curve[:, 1], 'ko--', label='control string')
plt.plot(lin[:, 0], lin[:, 1], '-', label='linear reparam')
plt.plot(cub[:, 0], cub[:, 1], '-', label='cubic reparam')
plt.axis('equal')
plt.legend()
plt.title('Rn cubic vs linear')
plt.show()

plt.figure()
plt.plot(seg_lin.numpy(), label='linear segment lengths')
plt.plot(seg_cub.numpy(), label='cubic segment lengths')
plt.legend()
plt.title('Segment lengths after reparametrization')
plt.xlabel('segment index')
plt.ylabel('length')
plt.show()

## 4) SO(3) linear reparametrization
Create a non-uniform rotation sequence and verify relative angles become near-uniform.

In [ ]:
axis = torch.tensor([0.3, -0.7, 0.6])
axis = axis / axis.norm()
angles = torch.tensor([0.0, 0.15, 1.8, 2.0, 2.6])
rot_string = rotvec_to_rotmat(angles.unsqueeze(-1) * axis)
rot_new = uniform_string_repametrize_so3_linear(rot_string, n_new=60)

orig_ang = segment_angles_so3(rot_string)
new_ang = segment_angles_so3(rot_new)
print_stats('Original SO(3) relative angles', orig_ang)
print_stats('Resampled SO(3) relative angles', new_ang)

cum_orig = torch.cat([torch.zeros(1), torch.cumsum(orig_ang, dim=0)])
cum_new = torch.cat([torch.zeros(1), torch.cumsum(new_ang, dim=0)])

plt.figure()
plt.plot(cum_orig.numpy(), 'o--', label='original cumulative angle')
plt.plot(cum_new.numpy(), '.-', label='resampled cumulative angle')
plt.title('SO(3): cumulative rotation progress')
plt.xlabel('point index')
plt.ylabel('cumulative angle [rad]')
plt.legend()
plt.show()

plt.figure()
plt.plot(orig_ang.numpy(), 'o-', label='original relative angle')
plt.plot(new_ang.numpy(), '.-', label='resampled relative angle')
plt.title('SO(3): relative rotation angle per segment')
plt.xlabel('segment index')
plt.ylabel('angle [rad]')
plt.legend()
plt.show()

## 5) SE(3) linear reparametrization
Test joint translation/rotation resampling and show effect of `corr_coeff`.

In [ ]:
t = torch.tensor([0.0, 0.2, 0.55, 0.7, 1.0])
translations = torch.stack([
    torch.cos(2.0 * torch.pi * t),
    torch.sin(2.0 * torch.pi * t),
    t,
], dim=1)

yaw = torch.tensor([0.0, 0.2, 1.8, 2.0, 3.0])
rotations = rotvec_to_rotmat(torch.stack([torch.zeros_like(yaw), torch.zeros_like(yaw), yaw], dim=1))

tr0, rot0 = uniform_string_repametrize_se3_linear(translations, rotations, n_new=80, corr_coeff=0.0)
tr1, rot1 = uniform_string_repametrize_se3_linear(translations, rotations, n_new=80, corr_coeff=1.0)
tr3, rot3 = uniform_string_repametrize_se3_linear(translations, rotations, n_new=80, corr_coeff=3.0)

seg_t0 = segment_lengths_rn(tr0)
seg_t1 = segment_lengths_rn(tr1)
seg_t3 = segment_lengths_rn(tr3)
seg_r0 = segment_angles_so3(rot0)
seg_r1 = segment_angles_so3(rot1)
seg_r3 = segment_angles_so3(rot3)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(translations[:, 0], translations[:, 1], 'ko--', label='original')
ax[0].plot(tr0[:, 0], tr0[:, 1], label='corr=0.0')
ax[0].plot(tr1[:, 0], tr1[:, 1], label='corr=1.0')
ax[0].plot(tr3[:, 0], tr3[:, 1], label='corr=3.0')
ax[0].set_title('SE(3) translation trajectory (xy)')
ax[0].axis('equal')
ax[0].legend()

ax[1].plot(seg_t0.numpy(), label='trans seg len corr=0.0')
ax[1].plot(seg_t1.numpy(), label='trans seg len corr=1.0')
ax[1].plot(seg_t3.numpy(), label='trans seg len corr=3.0')
ax[1].set_title('Translation segment lengths')
ax[1].set_xlabel('segment index')
ax[1].set_ylabel('length')
ax[1].legend()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(seg_r0.numpy(), label='rotation seg angle corr=0.0')
plt.plot(seg_r1.numpy(), label='rotation seg angle corr=1.0')
plt.plot(seg_r3.numpy(), label='rotation seg angle corr=3.0')
plt.title('Rotation segment angles')
plt.xlabel('segment index')
plt.ylabel('angle [rad]')
plt.legend()
plt.show()

## 6) Degenerate edge cases
Single-point strings should expand to repeated values.

In [ ]:
p = torch.tensor([[1.2, -0.3]])
p_new = uniform_string_repametrize_rn_linear(p, 5)

R = torch.eye(3).unsqueeze(0)
R_new = uniform_string_repametrize_so3_linear(R, 5)

T = torch.tensor([[0.0, 0.0, 0.0]])
TR_new, RR_new = uniform_string_repametrize_se3_linear(T, R, 5)

print('Rn shape:', tuple(p_new.shape), '| all equal:', torch.allclose(p_new, p_new[0:1].expand_as(p_new)))
print('SO3 shape:', tuple(R_new.shape), '| all equal:', torch.allclose(R_new, R_new[0:1].expand_as(R_new)))
print('SE3 transl shape:', tuple(TR_new.shape), '| all equal:', torch.allclose(TR_new, TR_new[0:1].expand_as(TR_new)))
print('SE3 rot shape:', tuple(RR_new.shape), '| all equal:', torch.allclose(RR_new, RR_new[0:1].expand_as(RR_new)))

## 7) Why cubic can look worse on coarse paths
On sparse control points with sharp direction changes, cubic Hermite interpolation can overshoot between points. Linear interpolation cannot overshoot because it stays on each segment.

In [ ]:
coarse = torch.tensor([[0.0, 0.0], [0.5, 1.0], [1.3, 0.2], [2.0, 1.2], [3.0, 0.8]], dtype=torch.float64)
lin_c = uniform_string_repametrize_rn_linear(coarse, n_new=120)
cub_c = uniform_string_repametrize_rn_cubic(coarse, n_new=120)

plt.figure()
plt.plot(coarse[:, 0], coarse[:, 1], 'ko--', label='coarse control points')
plt.plot(lin_c[:, 0], lin_c[:, 1], label='linear')
plt.plot(cub_c[:, 0], cub_c[:, 1], label='cubic')
plt.axis('equal')
plt.legend()
plt.title('Coarse path: cubic may overshoot')
plt.show()

print_stats('Linear seg lengths (coarse)', segment_lengths_rn(lin_c))
print_stats('Cubic seg lengths (coarse)', segment_lengths_rn(cub_c))

## 8) Smooth-path comparison against ground truth
Sample a smooth curve sparsely, reparametrize, then compare both methods to a dense ground-truth curve by nearest-point distance.

In [ ]:
def smooth_curve(u):
    x = u
    y = 0.25 * torch.sin(2 * torch.pi * u) + 0.08 * torch.sin(6 * torch.pi * u)
    return torch.stack([x, y], dim=1)

u_ctrl = torch.linspace(0.0, 1.0, 14, dtype=torch.float64)
u_dense = torch.linspace(0.0, 1.0, 2500, dtype=torch.float64)
ctrl = smooth_curve(u_ctrl)
truth = smooth_curve(u_dense)

lin_s = uniform_string_repametrize_rn_linear(ctrl, n_new=220)
cub_s = uniform_string_repametrize_rn_cubic(ctrl, n_new=220)

def nearest_error(points, reference):
    d = torch.cdist(points, reference)
    return d.min(dim=1).values

err_lin = nearest_error(lin_s, truth)
err_cub = nearest_error(cub_s, truth)

print_stats('Nearest-point error linear', err_lin)
print_stats('Nearest-point error cubic', err_cub)
print('Mean error ratio cubic/linear:', (err_cub.mean() / err_lin.mean()).item())

plt.figure()
plt.plot(truth[:, 0], truth[:, 1], 'k-', linewidth=2, label='ground truth (dense)')
plt.plot(ctrl[:, 0], ctrl[:, 1], 'ko', markersize=4, label='coarse samples')
plt.plot(lin_s[:, 0], lin_s[:, 1], label='linear reparam')
plt.plot(cub_s[:, 0], cub_s[:, 1], label='cubic reparam')
plt.axis('equal')
plt.legend()
plt.title('Smooth path reconstruction')
plt.show()

plt.figure(figsize=(8, 3.5))
plt.plot(err_lin.numpy(), label='linear nearest-point error')
plt.plot(err_cub.numpy(), label='cubic nearest-point error')
plt.title('Error to smooth ground truth')
plt.xlabel('resampled point index')
plt.ylabel('distance')
plt.legend()
plt.show()

seg_lin_s = segment_lengths_rn(lin_s)
seg_cub_s = segment_lengths_rn(cub_s)
print_stats('Smooth path linear segment lengths', seg_lin_s)
print_stats('Smooth path cubic segment lengths', seg_cub_s)

## 9) SE(3) edge cases: translation-only and rotation-only
Verify limiting behaviors visually and numerically.

In [ ]:
# Translation-only case
trans_only = torch.tensor([[0.0, 0.0, 0.0], [0.1, 0.0, 0.0], [1.1, 0.5, 0.0], [2.0, 1.5, 0.2]], dtype=torch.float64)
rot_const = torch.eye(3, dtype=torch.float64).unsqueeze(0).expand(trans_only.shape[0], 3, 3).clone()
new_t_to, new_r_to = uniform_string_repametrize_se3_linear(trans_only, rot_const, n_new=50, corr_coeff=2.0)

# Rotation-only case
zeros_t = torch.zeros((5, 3), dtype=torch.float64)
axis = torch.tensor([0.7, -0.4, 0.2], dtype=torch.float64)
axis = axis / axis.norm()
ang = torch.tensor([0.0, 0.1, 1.0, 1.7, 2.6], dtype=torch.float64)
rot_only = rotvec_to_rotmat(ang.unsqueeze(-1) * axis)
new_t_ro, new_r_ro = uniform_string_repametrize_se3_linear(zeros_t, rot_only, n_new=50, corr_coeff=1.0)

print('translation-only: rotation stayed constant =', torch.allclose(new_r_to, new_r_to[:1].expand_as(new_r_to), atol=1e-12))
print('rotation-only: translation stayed zero =', torch.allclose(new_t_ro, torch.zeros_like(new_t_ro), atol=1e-12))

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(trans_only[:, 0], trans_only[:, 1], 'ko--', label='original trans-only')
plt.plot(new_t_to[:, 0], new_t_to[:, 1], label='SE(3) reparam')
plt.axis('equal')
plt.title('Translation-only (xy)')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(segment_angles_so3(rot_only).numpy(), 'o-', label='orig rel angles')
plt.plot(segment_angles_so3(new_r_ro).numpy(), '.-', label='reparam rel angles')
plt.title('Rotation-only relative angles')
plt.xlabel('segment index')
plt.ylabel('angle [rad]')
plt.legend()
plt.tight_layout()
plt.show()

## 10) SE(3) edge case: tiny repeated steps and merge tolerance
Construct nearly-duplicate neighboring poses and inspect impact of `merge_tol`.

In [ ]:
trans_m = torch.tensor([
    [0.0, 0.0, 0.0],
    [1e-6, 0.0, 0.0],
    [2e-6, 0.0, 0.0],
    [0.8, 0.2, 0.1],
    [1.6, 0.5, 0.2],
], dtype=torch.float64)
yaw_m = torch.tensor([0.0, 1e-6, 2e-6, 1.2, 2.0], dtype=torch.float64)
rot_m = rotvec_to_rotmat(torch.stack([torch.zeros_like(yaw_m), torch.zeros_like(yaw_m), yaw_m], dim=1))

t_no_merge, r_no_merge = uniform_string_repametrize_se3_linear(trans_m, rot_m, n_new=60, corr_coeff=1.0, merge_tol=None)
t_merge, r_merge = uniform_string_repametrize_se3_linear(trans_m, rot_m, n_new=60, corr_coeff=1.0, merge_tol=1e-3)

comb_no = segment_lengths_rn(t_no_merge) + segment_angles_so3(r_no_merge)
comb_me = segment_lengths_rn(t_merge) + segment_angles_so3(r_merge)
print_stats('Combined cost no-merge', comb_no)
print_stats('Combined cost with-merge', comb_me)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(trans_m[:, 0], trans_m[:, 1], 'ko--', label='original')
plt.plot(t_no_merge[:, 0], t_no_merge[:, 1], label='no merge')
plt.plot(t_merge[:, 0], t_merge[:, 1], label='merge_tol=1e-3')
plt.axis('equal')
plt.title('Trajectory effect of merge_tol')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(comb_no.numpy(), label='combined seg cost no-merge')
plt.plot(comb_me.numpy(), label='combined seg cost with-merge')
plt.title('Segment combined costs')
plt.xlabel('segment index')
plt.ylabel('cost')
plt.legend()
plt.tight_layout()
plt.show()

## 11) SE(3) edge case: very large `corr_coeff`
Show transition from translation-dominant to rotation-dominant parametrization.

In [ ]:
t = torch.tensor([0.0, 0.2, 0.45, 0.8, 1.0], dtype=torch.float64)
translations = torch.stack([1.5 * t, 0.2 * torch.sin(4 * torch.pi * t), t**2], dim=1)
angles = torch.tensor([0.0, 0.15, 1.4, 1.7, 2.9], dtype=torch.float64)
rotations = rotvec_to_rotmat(torch.stack([torch.zeros_like(angles), torch.zeros_like(angles), angles], dim=1))

coeffs = [0.0, 0.5, 2.0, 20.0]
res = [uniform_string_repametrize_se3_linear(translations, rotations, n_new=80, corr_coeff=c) for c in coeffs]

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(translations[:, 0], translations[:, 1], 'ko--', label='original')
for c, (tt, _) in zip(coeffs, res):
    plt.plot(tt[:, 0], tt[:, 1], label=f'corr={c}')
plt.title('SE(3): translation projection (xy)')
plt.axis('equal')
plt.legend()

plt.subplot(1, 2, 2)
for c, (_, rr) in zip(coeffs, res):
    plt.plot(segment_angles_so3(rr).numpy(), label=f'corr={c}')
plt.title('SE(3): rotation relative angles')
plt.xlabel('segment index')
plt.ylabel('angle [rad]')
plt.legend()
plt.tight_layout()
plt.show()

for c, (tt, rr) in zip(coeffs, res):
    comb = segment_lengths_rn(tt) + c * segment_angles_so3(rr)
    print_stats(f'combined segment costs corr={c}', comb)

## 12) SE(3) endpoint and orthogonality sanity checks
Check endpoint preservation and that output rotations stay in SO(3).

In [ ]:
translations = torch.tensor([[0.0, 0.0, 0.0], [0.3, 0.1, -0.2], [0.7, 0.6, 0.1], [1.0, 1.0, 0.2]], dtype=torch.float64)
rv = torch.tensor([[0.0, 0.0, 0.0], [0.0, 0.3, 0.2], [0.1, 0.5, 0.4], [0.3, 0.8, 0.9]], dtype=torch.float64)
rotations = rotvec_to_rotmat(rv)
new_t, new_r = uniform_string_repametrize_se3_linear(translations, rotations, n_new=120, corr_coeff=1.0)

print('start translation preserved:', torch.allclose(new_t[0], translations[0], atol=1e-12))
print('end translation preserved:  ', torch.allclose(new_t[-1], translations[-1], atol=1e-12))
print('start rotation preserved:   ', torch.allclose(new_r[0], rotations[0], atol=1e-12))
print('end rotation preserved:     ', torch.allclose(new_r[-1], rotations[-1], atol=1e-12))

I = torch.eye(3, dtype=torch.float64)
orth_err = torch.linalg.norm(new_r.transpose(-1, -2) @ new_r - I, dim=(-2, -1))
det_err = torch.abs(torch.det(new_r) - 1.0)
print_stats('Orthogonality error ||R^T R - I||_F', orth_err)
print_stats('Determinant error |det(R)-1|', det_err)

plt.figure(figsize=(8, 3.5))
plt.plot(orth_err.numpy(), label='orthogonality error')
plt.plot(det_err.numpy(), label='determinant error')
plt.yscale('log')
plt.xlabel('point index')
plt.title('SO(3) numerical sanity along SE(3) output')
plt.legend()
plt.show()

## 13) SE(3) 3D visualization: translations and rotations (axis-angle)
Two separate 3D plots: one for translation trajectories in Cartesian space, and one for rotation trajectories in axis-angle (rotation-vector) space.

In [ ]:
from diffusion_strings.so3 import rotmat_to_rotvec

# Build a non-trivial SE(3) path
u = torch.tensor([0.0, 0.1, 0.35, 0.6, 0.8, 1.0], dtype=torch.float64)
translations = torch.stack([
    1.2 * torch.cos(2.0 * torch.pi * u),
    0.9 * torch.sin(2.0 * torch.pi * u),
    1.8 * u - 0.2 * torch.sin(4.0 * torch.pi * u),
], dim=1)

rv = torch.stack([
    0.8 * u,
    0.5 * torch.sin(2.0 * torch.pi * u),
    1.6 * u**2,
], dim=1)
rotations = rotvec_to_rotmat(rv)

coeffs = [0.0, 1.0, 4.0]
results = [uniform_string_repametrize_se3_linear(translations, rotations, n_new=120, corr_coeff=c) for c in coeffs]

# Convert resampled rotations to axis-angle vectors for plotting
rotvec_results = [rotmat_to_rotvec(r) for (_, r) in results]
rotvec_orig = rotmat_to_rotvec(rotations)

In [ ]:
# 3D plot: translations
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')

ax.plot(
    translations[:, 0].numpy(), translations[:, 1].numpy(), translations[:, 2].numpy(),
    'ko--', label='original (coarse)'
)
for c, (t_new, _) in zip(coeffs, results):
    ax.plot(t_new[:, 0].numpy(), t_new[:, 1].numpy(), t_new[:, 2].numpy(), label=f'reparam corr={c}')

ax.set_title('SE(3) reparametrization: translations in 3D')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_zlabel('z')
ax.legend()
plt.show()

In [ ]:
# 3D plot: rotations in axis-angle (rotation-vector) space
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')

ax.plot(
    rotvec_orig[:, 0].numpy(), rotvec_orig[:, 1].numpy(), rotvec_orig[:, 2].numpy(),
    'ko--', label='original (coarse)'
)
for c, rv_new in zip(coeffs, rotvec_results):
    ax.plot(rv_new[:, 0].numpy(), rv_new[:, 1].numpy(), rv_new[:, 2].numpy(), label=f'reparam corr={c}')

ax.set_title('SE(3) reparametrization: rotations in axis-angle space')
ax.set_xlabel('rotvec x')
ax.set_ylabel('rotvec y')
ax.set_zlabel('rotvec z')
ax.legend()
plt.show()